In [ ]:
import numpy as np
import napari
import pandas as pd

from skimage import io
from vispy.color import get_colormap
import trimesh

from morphotrack.analysis import tracks_to_vectors

In [ ]:
# load tracks
tracks_asma = pd.read_csv("path_to_bigtrace.xml_traces.csv")
scale = np.asarray((2.0,1.3,1.3)) # Set scale to micron to be consistency

roi_num_col = "ROI_Number"
coord_cols = ['Z_coord','Y_coord','X_coord']

# load meshes for scale checking
mesh01 = trimesh.load("../data/mesh_wm.ply")
mesh02 = trimesh.load("../data/mesh_pia.ply")

# confirm the order of the points is from out to in.
points_asma = tracks_asma[coord_cols].to_numpy() * scale
track_ids = tracks_asma[roi_num_col].to_numpy()

# dense: all points; sparse: first/last segment only
dense_option = True
track_positions, local_vectors = tracks_to_vectors(points_asma, track_ids, dense=dense_option)

In [ ]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_vectors(np.stack([track_positions, local_vectors], axis=1), length=10, edge_width=20, edge_color='red', name='3D Vectors',out_of_slice_display=True)
viewer.add_surface((mesh01.vertices, mesh01.faces), colormap="magenta", blending="additive", shading="smooth")
viewer.add_surface((mesh02.vertices, mesh02.faces), colormap="green", blending="additive", shading="smooth")

In [ ]:
suffix = "dense" if dense_option else "sparse"
np.savez_compressed(
    f"../data/tracks_{suffix}.npz",
    positions=track_positions,
    vectors=local_vectors,
    unit="micron"
)